# Notebook 01 — IFC Graph Construction

Builds the typed IFC property graph G=(V,E,τ) from the reference IFC file, inspects statistics, and saves the graph for reuse.

**Prerequisites:** Run Notebook 00 first to download `duplex.ifc`.
**No GPU required.**

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import os, sys
REPO = 'ifc-graphrag-dt'

if os.path.exists(REPO):
    !cd {REPO} && git pull --quiet
else:
    !git clone https://github.com/aiwithprashant/ifc-graphrag-dt.git

!bash {REPO}/colab_setup.sh

os.chdir(REPO)
REPO = '.'   # paths after chdir must not include repo name
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working directory: {os.getcwd()}')

import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

from pipeline.layer1_retriever.ifc_graph_builder import IFCGraphBuilder
from pipeline.layer1_retriever.khop_traversal import KHopTraversal

IFC_PATH  = 'benchmark/ifc_reference_models/duplex.ifc'
GRAPH_OUT = 'outputs/graphs/ifc_graph.json'
os.makedirs('outputs/graphs',   exist_ok=True)
os.makedirs('outputs/figures',  exist_ok=True)
print('Imports OK')

In [ ]:
# ── Cell 2: Verify IFC file exists ────────────────────────────────────────────
if not Path(IFC_PATH).exists():
    import urllib.request
    print('Downloading duplex.ifc...')
    DUPLEX_URL = 'https://github.com/buildingSMART/Sample-Test-Files/raw/master/IFC%202x3/Duplex%20Apartment/Duplex_A_20110907.ifc'
    urllib.request.urlretrieve(DUPLEX_URL, IFC_PATH)
    print(f'Downloaded: {Path(IFC_PATH).stat().st_size/1024:.1f} KB')
else:
    print(f'IFC file ready: {IFC_PATH} ({Path(IFC_PATH).stat().st_size/1024:.1f} KB)')

In [ ]:
# ── Cell 3: Build or load graph ───────────────────────────────────────────────
if Path(GRAPH_OUT).exists():
    print('Loading cached IFC graph...')
    G = IFCGraphBuilder.load_graph(GRAPH_OUT)
else:
    print('Building IFC graph from file (takes ~30s)...')
    builder = IFCGraphBuilder(IFC_PATH)
    G = builder.build()
    builder.save_graph(GRAPH_OUT)
    print(f'Graph saved: {GRAPH_OUT}')

print(f'\nGraph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

In [ ]:
# ── Cell 4: Entity and relation type distributions ────────────────────────────
node_types = Counter(d.get('ifc_type','?') for _, d in G.nodes(data=True))
edge_types = Counter(d.get('relation_type','?') for _, _, d in G.edges(data=True))

print('TOP 15 ENTITY TYPES')
print(f'{"Type":<45} {"Count":>6}')
print('-' * 55)
for t, n in node_types.most_common(15):
    print(f'{t:<45} {n:>6}')

print('\nRELATION TYPE DISTRIBUTION')
print(f'{"Relation":<48} {"Count":>6}')
print('-' * 58)
for r, n in edge_types.most_common():
    print(f'{r:<48} {n:>6}')

In [ ]:
# ── Cell 5: Entity type distribution chart ────────────────────────────────────
top_n = 20
top_types = node_types.most_common(top_n)
labels = [t[0].replace('Ifc','') for t in top_types]
values = [t[1] for t in top_types]

plt.figure(figsize=(14, 5))
plt.barh(labels[::-1], values[::-1], color='#2E5FA3')
plt.xlabel('Instance Count')
plt.title(f'Top {top_n} IFC Entity Types in Reference Model')
plt.tight_layout()
plt.savefig('outputs/figures/01_entity_type_distribution.png', dpi=150)
plt.show()
print('Figure saved: outputs/figures/01_entity_type_distribution.png')

In [ ]:
# ── Cell 6: Degree distribution ───────────────────────────────────────────────
in_degrees  = [d for _, d in G.in_degree()]
out_degrees = [d for _, d in G.out_degree()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(in_degrees,  bins=30, color='#1D9E75', edgecolor='white')
axes[0].set_title('In-degree Distribution')
axes[0].set_xlabel('In-degree')
axes[0].set_ylabel('Node count')

axes[1].hist(out_degrees, bins=30, color='#D85A30', edgecolor='white')
axes[1].set_title('Out-degree Distribution')
axes[1].set_xlabel('Out-degree')

plt.tight_layout()
plt.savefig('outputs/figures/01_degree_distribution.png', dpi=150)
plt.show()
print(f'Max in-degree: {max(in_degrees)}  |  Max out-degree: {max(out_degrees)}')
print(f'Mean in-degree: {sum(in_degrees)/len(in_degrees):.2f}  |  Mean out-degree: {sum(out_degrees)/len(out_degrees):.2f}')

In [ ]:
# ── Cell 7: k-hop traversal depth analysis ────────────────────────────────────
# Find IfcSpace nodes to use as seeds
space_nodes = [n for n, d in G.nodes(data=True) if d.get('ifc_type') == 'IfcSpace']
print(f'Found {len(space_nodes)} IfcSpace nodes in graph')

if not space_nodes:
    # Fallback: use any node with most connections
    space_nodes = [max(G.nodes(), key=lambda n: G.degree(n))]
    print(f'Fallback seed: {space_nodes[0]}')

print('\nTraversal depth analysis — demonstrates k>=3 needed for Tier 3:')
print(f'{"Depth":<8} {"Nodes":>8} {"Edges":>8} {"Relation types":>16}')
print('-' * 50)
for depth in [1, 2, 3, 4]:
    trav = KHopTraversal(G, max_depth=depth, bidirectional=True)
    r = trav.traverse([space_nodes[0]])
    rel_types = len(set(r.relation_type_counts.keys()))
    print(f'{depth:<8} {r.node_count:>8} {r.edge_count:>8} {rel_types:>16}')

print('\nConclusion: Tier 3 system prompts require depth>=3 to')
print('recover containment + connectivity + system membership chains.')
print('\nGraph construction complete. Proceed to Notebook 02.')